# HemoLink ML — EDA, Modelagem Comparativa, Calibração e Explicabilidade (SHAP)

**Objetivo**: Construir e validar um sistema de **Machine Learning em Python** para encontrar e ranquear os doadores de sangue mais correspondentes para um pedido alvo.

### Arquitetura Híbrida em Dois Estágios
1. **Estágio 1 — Filtro Clínico Determinístico (`ml/compatibility.py`)**: garante 100% de segurança biológica (matriz ABO/Rh por hemocomponente: Hemácias, Plasma, Plaquetas, Crioprecipitado) e elegibilidade ANVISA/Ministério da Saúde.
2. **Estágio 2 — Motor de Machine Learning (`ml/train.py`, `ml/predictor.py`)**: entre os doadores elegíveis, prevê a probabilidade calibrada de efetivação da doação ($P(y=1 \mid \mathbf{x})$), ordena o ranking Top-K para a tela de Revisão Clínica e gera explicações locais com **SHAP**.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SERVER_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(SERVER_DIR) not in sys.path:
    sys.path.insert(0, str(SERVER_DIR))

from ml.compatibility import BLOOD_TYPE_PREVALENCE_BR, compatible_donor_types
from ml.data_generator import UCI_TRANSFUSION_REFERENCE_STATS, generate_demo_donors
from ml.predictor import matcher_service
from ml.train import DATASET_PATH, METRICS_REPORT_PATH, PLOTS_DIR, run_training_pipeline

print('Diretório do servidor:', SERVER_DIR)

## 1. Validação do Estágio 1 (Compatibilidade ABO/Rh por Hemocomponente)
Demonstra a diferença clínica entre **Concentrado de hemácias** e **Plasma fresco congelado** (regra ABO inversa).

In [ ]:
for comp in ['Concentrado de hemácias', 'Plasma fresco congelado', 'Concentrado de plaquetas']:
    print(f'\n=== {comp} ===')
    for bt in ['O-', 'O+', 'A+', 'AB+']:
        print(f'  Receptor {bt:>3} <- Doadores compatíveis: {sorted(compatible_donor_types(bt, comp))}')

## 2. Carregamento do Dataset e Análise Exploratória (EDA)
Carrega o dataset gerado pelo simulador controlado (`training_dataset.csv`) — ou executa o pipeline de treino caso os artefatos ainda não existam — e compara estatísticas RFM com o dataset público *UCI Blood Transfusion Service Center*.

In [ ]:
if not DATASET_PATH.exists() or not METRICS_REPORT_PATH.exists():
    run_training_pipeline(save_artifacts=True)

df = pd.read_csv(DATASET_PATH)
report = json.loads(METRICS_REPORT_PATH.read_text(encoding='utf-8'))

print(f'Shape do dataset: {df.shape}')
print(f'Pedidos únicos: {df["pedido_id"].nunique()} | Taxa de sucesso (y=1): {df["match_sucesso"].mean():.1%}')
print(f'Doadores novos (cold start): {df["eh_doador_novo"].mean():.1%}')
print('Referência UCI Blood Transfusion:', UCI_TRANSFUSION_REFERENCE_STATS)
df.head(5)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# Taxa de match por urgência e faixa de distância
df['faixa_dist'] = pd.cut(df['distancia_km'], bins=[0, 5, 12, 25, 60, 260], labels=['0-5km', '5-12km', '12-25km', '25-60km', '60km+'])
piv = df.pivot_table(index='faixa_dist', columns='urgencia_nivel', values='match_sucesso', aggfunc='mean', observed=False)
piv.columns = ['Normal', 'Urgente', 'Emergência']
piv.plot(kind='bar', ax=axes[0], color=['#6B6B73', '#E07A5F', '#C8102E'])
axes[0].set_title('Taxa de Match: Distância × Urgência')
axes[0].set_ylabel('Proporção de Matches (y=1)')
axes[0].tick_params(axis='x', rotation=20)

# Taxa de match por situação de disponibilidade
df.groupby('situacao_disponibilidade')['match_sucesso'].mean().sort_values().plot(
    kind='barh', ax=axes[1], color='#C8102E'
)
axes[1].set_title('Taxa de Match por Disponibilidade')
axes[1].set_xlabel('Proporção de Matches (y=1)')

# Distribuição de tipos sanguíneos dos doadores elegíveis
df['tipo_doador'].value_counts(normalize=True).plot(kind='bar', ax=axes[2], color='#2B2B30')
axes[2].set_title('Frequência de Tipos Sanguíneos na Base')
axes[2].set_ylabel('Proporção')

plt.tight_layout()
plt.show()

## 3. Comparativo Experimental de Modelos (GroupSplit + GroupKFold)
Avaliamos 6 abordagens no mesmo conjunto de teste isolado agrupado por `pedido_id` (zero *data leakage*): Baseline Heurístico por Distância, Regressão Logística, Random Forest, LightGBM, **LightGBM Calibrado (Produção)** e `LGBMRanker` (LambdaRank).

In [ ]:
metrics_df = pd.DataFrame(report['test_metrics']).T
display(metrics_df[['roc_auc', 'pr_auc', 'f1', 'brier_score', 'ndcg_at_5', 'precision_at_3', 'recall_at_5', 'mrr']])

## 4. Visualizações de Avaliação, Calibração e Explicabilidade (SHAP)

In [ ]:
from IPython.display import Image, display

for plot_name in ['model_comparison.png', 'roc_pr_curves.png', 'calibration_curve.png', 'shap_summary.png', 'shap_waterfall_example.png']:
    path = PLOTS_DIR / plot_name
    if path.exists():
        print(f'--- {plot_name} ---')
        display(Image(filename=str(path)))

## 5. Simulação de Inferência Online — Pedido `#2041` (Hospital Santa Clara, `O-`, Emergência)
Demonstra exatamente o ranking e as justificativas explicáveis enviadas à tela `RevisaoClinicaPage.jsx`.

In [ ]:
pedido_2041 = {
    'tipo_sanguineo': 'O-',
    'componente': 'Concentrado de hemácias',
    'urgencia': 'emergencia',
    'bolsas_total': 5,
    'bolsas_recebidas': 1,
    'estado': 'PE',
    'cidade': 'Recife',
    'hospital': 'Hospital Santa Clara',
}

resultado = matcher_service.rank_donors(pedido_2041, generate_demo_donors(), top_k=5)
print(f"Modelo: {resultado['modelo']} (v{resultado['versaoModelo']})")
print(f"Doadores avaliados: {resultado['totalAvaliados']} | Elegíveis após Estágio 1: {resultado['totalElegiveis']}")
pd.DataFrame(resultado['candidatos'])[['ranking', 'codigo', 'cidade', 'tipoSanguineo', 'distanciaKm', 'compatibilidade', 'situacao', 'selecionado', 'fatoresPrincipais']]